In [ ]:
input_data = None
targetpop_data = None
output_data = None
output_model = None
util = None
display_util = None
configfile = "config/config.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)
plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rule_setup,
    display_data_doc,
    display_long_data_doc,
    collist,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    SpenderID,
    drop_duplicate_columns,
    common_translate,
    split_data,
    collapse_col,
    find_redundant_cols,
    fix_redundancies,
    fix_units,
)

### Target Population Filtering

The donors in the dataset were filtered to match the target population (see [](general:tpf)). Afterwards we tried again to remove empty and duplicate columns.

In [ ]:
data = pd.read_parquet(input_data)
donors = collapse_col(
    data.loc[:, ["donor_et_dso", "donor_et_id_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
targetpop = pd.read_parquet(targetpop_data)
data = data[donors.isin(targetpop["donor_et_id_et"])]
display(
    Markdown(
        f"""The filter process reduced the number of donors in the data ({donors.nunique()}) and target population ({targetpop["donor_et_id_et"].nunique()})
            to {donors[donors.isin(targetpop["donor_et_id_et"])].nunique()} in the processed data.
        """
    )
)

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

### Integration of Seperated Institute Data

In this file, the IQTIG and ET data is not already connected (TODO are they?) (see [](general:ic)). The following table lists the different types of rows, whhich occur in this file and which ID combination they use.

In [ ]:
idcols = ["donor_et_dso", "donor_et_id_et"]
assert len(split_data(data, idcols)) == 3, "Not 3 different row types present!?"
assert (
    data.loc[:, ["date_et", "date_dso"]].diff(axis=1).iloc[:, 1].dropna() == 0
).all(), "Dates sometimes different"

## Domain Steps

For this file the general plan for domain preprocessing of longitudinal data was followed (see [](general:ds)).

### Row Filtering

There is no column differentiating between different types of monitoring (see [](general:rf)). The following analysis compares the data from the different sources. All rows were kept.

In [ ]:
data["Institute with a date"] = (
    (~data["date_dso"].isna()) + (~data["date_et"].isna()) * 2
).replace({1: "DSO", 2: "ET", 3: "DSO+ET", 0: "No Date"})
data["donor"] = collapse_col(
    data.loc[:, ["donor_et_dso", "donor_et_id_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
data["date"] = collapse_col(
    data.loc[:, ["date_dso", "date_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
display_long_data_doc(
    data,
    [
        "donor",
    ],
    "date",
    "Institute with a date",
)
data.drop(
    columns=["date", "donor", "Institute with a date"],
    inplace=True,
)

### Unit Conversions

First common translations were applied and then we converted different pressure measurements to a common unit (see [](general:uc)). Afterwards, unit specifier columns with only a single unit were removed.

In [ ]:
data = common_translate(data, config["data"]["common_translations"])

In [ ]:
fix_units(
    data,
    "hypotension_duration_min_et",
    "hypotension_duration_unit_et",
    config["data"]["unit_conversions"]["time_min"]["target"],
    config["data"]["unit_conversions"]["time_min"]["factors"],
)
fix_units(
    data,
    "central_venous_pressure_mm_hg_et",
    "central_venous_pressure_unit_et",
    config["data"]["unit_conversions"]["gas_mmhg"]["target"],
    config["data"]["unit_conversions"]["gas_mmhg"]["factors"],
)

In [ ]:
cols = data.columns[data.columns.to_series().str.contains("_unit")]
assert (data[cols].nunique() != 1).sum() == 0
dropme = cols[data[cols].nunique() == 1]
data = data.drop(columns=dropme)
display(
    Markdown(
        f"The columns {collist(dropme)} were removed as only a single unit was used."
    )
)

### Consolidating Columns

We consolidated columns that appear for {term}`ET` and {term}`DSO` (see [](general:crc))

In [ ]:
red = find_redundant_cols(data)
red["donor_et_id_et"] = ["donor_et_dso", "donor_et_id_et"]
fix_redundancies(data, red)

## Intermediate Dataset

For this longitudinal dataset we recommend the `date` column as the time axis.

In [ ]:
indcols = ["donor_et_id_et"]
data = data.sort_index(axis=1).sort_values(indcols + ["date"], axis=0)
data = data.set_index(indcols)

In [ ]:
# Another base class might be necessary, see util.py
# describe columns, without checks for now, order is important
class DonorPostmortemMonitoring(SpenderID):
    cardiac_arrest_min: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Cardiac arrest",
        description="How long did the patient experience cardiac arrest in min?",
    )
    central_venous_pressure_mm_hg: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Central venous pressure",
        description="What was the central venous pressure in mm Hg?",
    )
    communicated_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Communicated Date",
        description="When was the measurement comunicated?",
    )
    date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Measurement Date",
        description="When was the measurement taken?",
    )
    diastolic_blood_pressure_mm_hg: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Diastolic blood_pressure",
        description="What was the patients diastolic blood_pressure in mm Hg?",
    )
    diastolic_min_blood_pressure_mm_hg: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Minimum diastolic blood_pressure",
        description="What was the patients minimal diastolic blood_pressure in mm Hg?",
    )
    diuresis_interval_h: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Diuresis interval",
        description="What was the patients diuresis interval in h?",
    )
    diuresis_last_hour_ml: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Diuresis Amount last hour",
        description="What was the patients diuresis amount in the last hour in ml?",
    )
    diuresis_ml: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Diuresis Amount",
        description="What was the patients diuresis amount in ml?",
    )
    heart_bpm: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Heart",
        description="What was the patients heartrate amount in BPM?",
    )
    hypertension_duration_min: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hypertension duration",
        description="How long did the patient experience hypertension in min?",
    )
    hypotension_duration_min: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hypotentsion duration",
        description="How long did the patient experience hypotentsion in min?",
    )
    left_apex_diaphragma_cm: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Left Apex Diaphragma",
        description="How high (?) was the left diaphragma in cm?",
    )
    left_apex_left_cpa_cm: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Left Apex Diaphragma Costophrenic angle",
        description="How high (?) was the left diaphragma in cm at the costophrenic angle point?",
    )
    peakpressure_cm_h2o: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Peak Pressue",
        description="How high was the peak lung (?) pressure in cm H2O?",
    )
    right_apex_diaphragma_cm: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Right Apex Diaphragma",
        description="How high (?) was the right diaphragma in cm?",
    )
    right_apex_right_cpa_cm: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Right Apex Diaphragma Costophrenic angle",
        description="How high (?) was the right diaphragma in cm at the costophrenic angle point?",
    )
    right_cpa_left_cpa_cm: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Right to left Apex diaphragma Costophrenic angle",
        description="Distance between the right (?) and the left the costophrenic angle point?",
    )
    steatosis_hepatis: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Steatosis Hepatis",
        description="Was steatosis hepatis diagnosed?",
        isin=["yes", "no"],
    )
    systolic_blood_pressure_mm_hg: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Systolic blood_pressure",
        description="What was the patients systolic blood_pressure in mm Hg?",
    )
    systolic_min_blood_pressure_mm_hg: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Minimum systolic blood_pressure",
        description="What was the patients minimal systolic blood_pressure in mm Hg?",
    )
    temp_deg_celsius: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Temperature",
        description="What was the patients temperature in °C?",
    )
    thorax_circumference_cm: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Thorax Circumference",
        description="What was the patients thorax circumference in cm",
    )

    class Config:
        title = "Donor Postmortem Monitoring Dataset"
        description = "Each row represents a intensive care monitoring measurement. The data is based on the 'element_spender_postmortem_monitoring.csv' file. It contains data from the DSO and ET."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(DonorPostmortemMonitoring, data)

In [ ]:
DonorPostmortemMonitoring.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    DonorPostmortemMonitoring.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)